# NCS inference (GPU): run once per volume and model

*Copy of `RGT-Est_NCS_Inference_Colab.ipynb` set up for the AVO18 mid stack, so both volumes can run at once in separate Colab sessions. It differs in `SEGY_FILE` and adds the `OVERLAP_FILTER` setting and the `RUNS` list for tile-artefact tests.*

Runs one of the NCS seismic foundation models (Norsk Regnesentral; Ordoñez et al. 2025, arXiv 2603.23211; https://github.com/NorskRegnesentral/NCS_models) on the seismic crop and saves the 768-channel feature cube to Drive. **Analysis happens in `RGT-Est_NCS_Analysis_Colab.ipynb`**, which runs on a CPU runtime.

| `MODEL_KIND` | What it looks at | Default batch |
|---|---|---|
| `"2d"` | single sections (inlines with `dir0`) | 64 |
| `"2.5d"` | several 2D views through each point (the model config's views, or `INPUT_VIEWS`) | 32 |
| `"3d"` | true 3D crops | 4 |

- **Weights** are downloaded once per model into `<DRIVE_FOLDER>/models/` and loaded from there afterwards.
- **The cube** is saved as `<DRIVE_FOLDER>/outputs/<seismic>_ncs_<model>_<direction>_dens<D>[_ov<N>][_<filter>].zarr` (`_ov<N>` only when `NUM_OVERLAP` isn't the default 7, `_<filter>` only when `OVERLAP_FILTER` isn't the default `ramp`), with a `.json` record of the settings, GPU and model version. If it already exists, inference is skipped.
- **Several runs in one session:** set `RUNS` to a list of overrides, e.g. `[dict(OVERLAP_FILTER="center"), dict(NUM_OVERLAP=13, OVERLAP_FILTER="center"), dict(DIRECTION="dir90")]`. Each entry runs in turn with the other settings unchanged, and already-finished runs are skipped.
- **Overlap:** some embedding channels encode where a point sits inside its 224-sample crop, which shows as bands repeating with the crop step. More overlap (`NUM_OVERLAP = 12`) blends each point from ~7 crops at different positions, so that artefact averages out. It costs roughly 10x the default run.
- **Why the crop position matters (2.5D):** the model sees four slices at once (inline, crossline, two diagonals), but the crossline and diagonal slices pass through the crop's **centre trace**. A patch near the crop edge gets its cross-slice context from up to 112 traces away.
- **Overlap filter:** how overlapping crops are blended. `ramp` (default) weights each crop linearly towards its centre; `gaussian` and `exponential` favour the centre more; `center` takes each patch only from the crop whose centre is nearest, so the cross slices pass within ±16 traces of it at overlap 12 and ±8 at overlap 13. `center` costs nothing extra; overlap 13 is ~4x overlap 12 (crop spacing halves along both the section and time).
- **Direction:** `dir0` lays crops along crosslines and time (one crop per inline), so any residual artefact repeats along crossline. A `dir90` run lays them along inlines instead; combining the two (average or concatenate, z-scoring each first, as in the NCS paper's late fusion) suppresses what's left.
- **Normalisation:** the input volume is scaled to unit variance and clipped at ±3σ before inference, the same as the NCS pre-training data.

**Runtime:** **A100 GPU** recommended for 3D (Runtime → Change runtime type). Then **Runtime → Run all**.

## 1. Settings

In [ ]:
DATA_SOURCE  = "drive"
DRIVE_FOLDER = "/content/drive/MyDrive/PUBLIC"
DATA_FOLDER  = "/content"
SEGY_FILE    = "1988_AVO18.sgy"    # 18-degree partial stack, unconditioned (copy it into DRIVE_FOLDER)

MODEL_KIND   = "2.5d"            # "2d", "2.5d" or "3d"
DIRECTION    = "dir0"            # primary traversal: "dir0" (inline) or "dir90" (crossline)
INPUT_VIEWS  = None              # 2.5D only: e.g. ["dir0", "dir90"]; None = the model's own views
DENSIFY      = 4                 # embedding every 16/DENSIFY samples (4 → every 4; 8 → every 2, ~8x bigger)
DTYPE        = "float16"         # "float16" or "bfloat16" on GPU
BATCH        = None              # None = default for the model (64 / 32 / 4)
NUM_OVERLAP  = 12                # crops overlap by this many 16-sample patches (max 13). 7 = NCS default
                                 # (crops every 112 traces); 12 = every 32 traces, which averages out the
                                 # tile-position artefact seen in some channels (~10x the compute of 7)
CLIP_SIGMA   = 3.0               # input normalisation, as in NCS pre-training: unit variance, clip at ±3σ
NORMALIZATION = "std"
OVERLAP_FILTER = "ramp"          # blending of overlapping crops: "ramp" (default), "gaussian", "exponential",
                                 # "center" (each patch from the nearest-centred crop only) or "uniform"

# Several runs in one session: each dict overrides the settings above for one run (None = a single run).
# Set up for the tile-artefact tests, cheapest first; runs already in Drive are skipped.
RUNS = [dict(OVERLAP_FILTER="center"),                    # same cost as overlap 12 ramp
        dict(DIRECTION="dir90"),                          # crops laid along inlines; same cost
        dict(NUM_OVERLAP=13, OVERLAP_FILTER="center")]    # ~4x the cost of overlap 12

## 2. Set up

In [ ]:
!pip -q install segyio "NCS[inference] @ git+https://github.com/NorskRegnesentral/NCS_models"
import os, time, shutil, json
import numpy as np, torch, segyio
MODELS = {"2d": ("NorskRegnesentralSTI/NCS-v1-2d-base", 64), "2.5d": ("NorskRegnesentralSTI/NCS-v1-2.5d-base", 32),
          "3d": ("NorskRegnesentralSTI/NCS-v1-3d-base", 4)}
MODEL_ID, default_batch = MODELS[MODEL_KIND]
BATCH = BATCH or default_batch
TAG = MODEL_KIND.replace(".", "")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: Runtime → Change runtime type → A100 (or L4) GPU.")
GPU = torch.cuda.get_device_name(0)
print(f"PyTorch {torch.__version__} | GPU {GPU} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | model {MODEL_ID} | batch {BATCH}")

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src, dst = os.path.join(DRIVE_FOLDER, SEGY_FILE), os.path.join(DATA_FOLDER, SEGY_FILE)
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
        shutil.copyfile(src, dst); print("copied", SEGY_FILE)
SEGY_PATH = os.path.join(DATA_FOLDER, SEGY_FILE)
BASE = DRIVE_FOLDER if DATA_SOURCE == "drive" else DATA_FOLDER
stem = os.path.splitext(SEGY_FILE)[0]
with segyio.open(SEGY_PATH) as f:                       # geometry the way NCS reads it (bytes 189/193)
    print(f"seismic as NCS sees it: {len(f.ilines)} inlines × {len(f.xlines)} crosslines × {len(f.samples)} samples")

## 3. Model weights: downloaded once into Drive

If Colab asks to grant access to a secret (`HF_TOKEN`), "Don't grant" is fine: the models are public.

In [ ]:
MODEL_DIR = os.path.join(BASE, "models", MODEL_ID.split("/")[-1])
if os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("model weights from Drive:", MODEL_DIR)
else:
    from huggingface_hub import snapshot_download, HfApi
    tmp_model = "/content/_ncs_model_download"
    if os.path.exists(tmp_model): shutil.rmtree(tmp_model)
    t0 = time.time()
    snapshot_download(repo_id=MODEL_ID, local_dir=tmp_model)
    try:
        sha = HfApi().model_info(MODEL_ID).sha
    except Exception as e:
        sha = f"unknown ({type(e).__name__})"
    with open(os.path.join(tmp_model, "_source.txt"), "w") as fh:
        fh.write(f"source: https://huggingface.co/{MODEL_ID}\ncommit: {sha}\n"
                 f"downloaded (UTC): {time.strftime('%Y-%m-%d %H:%M', time.gmtime())}\n")
    for junk in (".cache", ".gitattributes"):
        jp = os.path.join(tmp_model, junk)
        if os.path.isdir(jp): shutil.rmtree(jp)
        elif os.path.isfile(jp): os.remove(jp)
    os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
    shutil.copytree(tmp_model, MODEL_DIR, dirs_exist_ok=True)
    print(f"downloaded {MODEL_ID} in {time.time()-t0:.0f} s → {MODEL_DIR}")
print(open(os.path.join(MODEL_DIR, "_source.txt")).read() if os.path.exists(os.path.join(MODEL_DIR, "_source.txt")) else "")

## 4. Run inference (skipped if the cube is already in Drive)

In [ ]:
BASE_SETTINGS = dict(DIRECTION=DIRECTION, NUM_OVERLAP=NUM_OVERLAP, OVERLAP_FILTER=OVERLAP_FILTER)


def feature_path(direction, num_overlap, overlap_filter):
    ov = "" if num_overlap == 7 else f"_ov{num_overlap}"                  # default runs keep their original names
    flt = "" if overlap_filter == "ramp" else f"_{overlap_filter}"
    path = os.path.join(BASE, "outputs", f"{stem}_ncs_{TAG}_{direction}_dens{DENSIFY}{ov}{flt}.zarr")
    legacy = os.path.join(BASE, "outputs", f"{stem}_ncs_{direction}_dens{DENSIFY}.zarr")   # first 2D run's name
    if MODEL_KIND == "2d" and not ov and not flt and not os.path.exists(path) and os.path.exists(legacy):
        path = legacy
    return path


def run_one(direction, num_overlap, overlap_filter):
    path = feature_path(direction, num_overlap, overlap_filter)
    if os.path.exists(path):
        print("feature cube already in Drive:", path)
        return path
    from NCS.inference import run_inference
    tmp = f"/content/{os.path.basename(path)}"
    if os.path.exists(tmp): shutil.rmtree(tmp)
    kw = dict(model_path=MODEL_DIR, input_path=SEGY_PATH, output_path=tmp, direction=direction, densify=DENSIFY,
              num_overlap_patches=num_overlap, overlap_filter=overlap_filter, batch_size=BATCH, device="cuda",
              dtype=DTYPE, clip_sigma=CLIP_SIGMA, normalization_mode=NORMALIZATION)
    if INPUT_VIEWS:
        views = list(INPUT_VIEWS)
        kw["input_views"] = views if direction in views else [direction] + views
    print(f"running {direction}, overlap {num_overlap}, filter {overlap_filter} ...")
    t0 = time.time()
    run_inference(**kw)
    minutes = (time.time() - t0) / 60
    print(f"inference took {minutes:.1f} min on {GPU}")
    shutil.copytree(tmp, path)
    shutil.rmtree(tmp)                                                    # keep /content free for the next run
    record = dict(model=MODEL_ID, model_dir=MODEL_DIR, segy=SEGY_FILE, direction=direction, input_views=INPUT_VIEWS,
                  densify=DENSIFY, dtype=DTYPE, batch=BATCH, num_overlap=num_overlap, overlap_filter=overlap_filter,
                  clip_sigma=CLIP_SIGMA, normalization=NORMALIZATION, gpu=GPU, minutes=round(minutes, 1),
                  utc=time.strftime("%Y-%m-%d %H:%M", time.gmtime()))
    with open(path + ".json", "w") as fh: json.dump(record, fh, indent=1)
    print("saved to", path)
    return path


FEAT_PATHS = []
for override in (RUNS or [{}]):
    s = {**BASE_SETTINGS, **override}
    FEAT_PATHS.append(run_one(s["DIRECTION"], s["NUM_OVERLAP"], s["OVERLAP_FILTER"]))
FEAT_PATH = FEAT_PATHS[-1]

## 5. Quick check

In [ ]:
import xarray as xr
for path in FEAT_PATHS:
    groups = sorted(d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d)))
    ds = xr.open_zarr(path, group=groups[0])
    print(os.path.basename(path), "| groups:", groups, "|", dict(ds["features"].sizes))
print("\nNext: open RGT-Est_NCS_Analysis_Colab.ipynb (a CPU runtime is enough).")